# 13 · Decoding: temperature, top-k, top-p, min-p and beam search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/13_decoding.ipynb) &nbsp; 📖 Concept deck: [Decoding strategies](https://nilesh-salpe.github.io/visual-ai/inference/11-decoding/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** the sampling pipeline a serving engine runs on every step (repetition penalty, temperature, top-k, top-p, min-p) and beam search, then measure how temperature changes what each filter keeps.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(z):
    z = np.asarray(z, dtype=float); e = np.exp(z - z.max()); return e / e.sum()

P5 = np.array([0.50, 0.25, 0.15, 0.06, 0.04])      # the worked example on the deck
LOGITS = np.log(P5)
print('P5 =', P5, '; helper softmax(z)')

## Task 1: Temperature

Return the probabilities `softmax(logits / T)`. For `T == 0` return the **greedy** distribution: probability 1 on the argmax, 0 elsewhere (vLLM treats any temperature below 1e-5 as greedy).
Dividing logits by `T` is the same as raising probabilities to the power `1/T` and renormalising.

In [ ]:
def apply_temperature(logits, T):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_apply_temperature(f):
    assert np.allclose(f(LOGITS, 1.0), P5), "T = 1 must return the original probabilities"
    q = P5 ** 2 / (P5 ** 2).sum()
    assert np.allclose(f(LOGITS, 0.5), q), "T = 0.5 must give p^2, renormalised"
    g = f(LOGITS, 0)
    assert np.allclose(g, [1, 0, 0, 0, 0]), "T = 0 must be greedy (one-hot on the argmax)"
    assert f(LOGITS, 2.0).max() < 0.5, "T > 1 must flatten the distribution"

check("apply_temperature", _test_apply_temperature, apply_temperature)

## Task 2: Top-k

Keep the `k` most likely tokens, set the rest to 0 and renormalise. `k == 0` means "off" (keep everything), as in vLLM.

In [ ]:
def top_k_filter(probs, k):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_top_k_filter(f):
    assert np.allclose(f(P5, 2), [2/3, 1/3, 0, 0, 0]), "top-2 of P5 is 0.50 and 0.25, renormalised to 2/3 and 1/3"
    assert np.allclose(f(P5, 0), P5), "k = 0 disables top-k"
    assert np.allclose(f(P5, 5), P5), "k = vocabulary size keeps everything"

check("top_k_filter", _test_top_k_filter, top_k_filter)

## Task 3: Top-p (nucleus)

Sort tokens by probability and keep the **smallest** prefix whose cumulative probability is at least `p`; the token that crosses the threshold is included. Zero the rest and renormalise.
Floating-point sums can land a hair below `p` (0.5 + 0.25 + 0.15 = 0.8999…), so compare with a small tolerance (1e-9).

In [ ]:
def top_p_filter(probs, p):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_top_p_filter(f):
    assert np.allclose(f(P5, 0.9), [0.5/0.9, 0.25/0.9, 0.15/0.9, 0, 0]), "top-p 0.9 keeps the first three tokens of P5 (0.50 + 0.25 + 0.15 = 0.90)"
    assert np.allclose(f(P5, 0.8), [0.5/0.9, 0.25/0.9, 0.15/0.9, 0, 0]), "top-p 0.8 also needs the third token: 0.75 < 0.8"
    assert np.allclose(f(P5, 0.5), [1, 0, 0, 0, 0]), "the top token alone reaches 0.5"
    assert np.allclose(f(P5, 1.0), P5), "top-p 1.0 keeps everything"

check("top_p_filter", _test_top_p_filter, top_p_filter)

## Task 4: Min-p

Keep tokens whose probability is at least `min_p × max(probs)`, zero the rest and renormalise. `min_p == 0` keeps everything.
The cut is **relative**: it keeps few tokens when the model is confident and many when it is not.

In [ ]:
def min_p_filter(probs, min_p):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 4: run after implementing
def _test_min_p_filter(f):
    kept = f(P5, 0.1)
    assert np.allclose(kept, np.r_[P5[:4], 0] / P5[:4].sum()), "min-p 0.1: threshold 0.05 keeps 0.50, 0.25, 0.15, 0.06 and drops 0.04"
    assert np.allclose(f(P5, 0.0), P5), "min-p 0 keeps everything"
    flat = np.full(10, 0.1)
    assert (f(flat, 0.5) > 0).sum() == 10, "a flat distribution keeps every token"

check("min_p_filter", _test_min_p_filter, min_p_filter)

## Task 5: Beam search

`step(seq)` returns a probability vector over the next token given the tuple `seq`. Starting from the empty sequence, keep the `width` partial sequences with the highest **total log-probability**, extend each by every token, and repeat for `length` steps.
Return the best `(sequence, logprob)` found. With `width = 1` this is greedy decoding.

In [ ]:
def beam_search(step, width, length):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 5: run after implementing
# A two-step toy model where greedy is fooled: token 0 looks best first, but every continuation after it is weak.
def toy_step(seq):
    if len(seq) == 0: return np.array([0.6, 0.4])
    if seq[0] == 0:   return np.array([0.5, 0.5])
    return np.array([0.9, 0.1])
def _test_beam_search(f):
    seq1, lp1 = f(toy_step, 1, 2)
    assert tuple(seq1) == (0, 0) and np.isclose(lp1, np.log(0.6 * 0.5)), f"width 1 is greedy: expected (0, 0) with p = 0.30, got {seq1}"
    seq2, lp2 = f(toy_step, 2, 2)
    assert tuple(seq2) == (1, 0) and np.isclose(lp2, np.log(0.4 * 0.9)), f"width 2 must find (1, 0) with p = 0.36, got {seq2}"

check("beam_search", _test_beam_search, beam_search)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Temperature
def apply_temperature_solution(logits, T):
    logits = np.asarray(logits, dtype=float)
    if T < 1e-5:
        out = np.zeros_like(logits); out[np.argmax(logits)] = 1.0; return out
    return softmax(logits / T)

check("apply_temperature (reference)", _test_apply_temperature, apply_temperature_solution)

In [ ]:
# @title Solution for Task 2: Top-k
def top_k_filter_solution(probs, k):
    probs = np.asarray(probs, dtype=float)
    if k <= 0 or k >= len(probs): return probs / probs.sum()
    keep = np.argsort(-probs)[:k]
    out = np.zeros_like(probs); out[keep] = probs[keep]; return out / out.sum()

check("top_k_filter (reference)", _test_top_k_filter, top_k_filter_solution)

In [ ]:
# @title Solution for Task 3: Top-p (nucleus)
def top_p_filter_solution(probs, p):
    probs = np.asarray(probs, dtype=float)
    order = np.argsort(-probs)
    cum = np.cumsum(probs[order])
    n = int(np.searchsorted(cum, p - 1e-9)) + 1          # first index where cum >= p, inclusive
    keep = order[:n]
    out = np.zeros_like(probs); out[keep] = probs[keep]; return out / out.sum()

check("top_p_filter (reference)", _test_top_p_filter, top_p_filter_solution)

In [ ]:
# @title Solution for Task 4: Min-p
def min_p_filter_solution(probs, min_p):
    probs = np.asarray(probs, dtype=float)
    out = np.where(probs >= min_p * probs.max(), probs, 0.0); return out / out.sum()

check("min_p_filter (reference)", _test_min_p_filter, min_p_filter_solution)

In [ ]:
# @title Solution for Task 5: Beam search
def beam_search_solution(step, width, length):
    beams = [((), 0.0)]
    for _ in range(length):
        cand = [(seq + (t,), lp + np.log(pt)) for seq, lp in beams for t, pt in enumerate(step(seq)) if pt > 0]
        beams = sorted(cand, key=lambda c: -c[1])[:width]
    return beams[0]

check("beam_search (reference)", _test_beam_search, beam_search_solution)

---
## Experiments
These use the reference solutions. First, how many tokens each filter keeps as temperature changes, on a 1,000-token Zipf-like distribution (a peaked head and a long tail). Then a CTRL-style repetition penalty, applied before temperature as vLLM does.

In [ ]:
V = 1000
base = np.log(1.0 / np.arange(1, V + 1) ** 1.1)            # Zipf-like logits
print(" T     top-p 0.9 keeps   min-p 0.1 keeps   top-k 40 keeps   mass outside the top 40")
for T in [0.5, 0.7, 1.0, 1.3, 1.6]:
    q = apply_temperature_solution(base, T)
    kp = (top_p_filter_solution(q, 0.9) > 0).sum(); km = (min_p_filter_solution(q, 0.1) > 0).sum()
    tail = np.sort(q)[::-1][40:].sum()
    print(f"{T:3.1f}   {kp:14d}   {km:15d}   {40:14d}   {tail:20.2f}")
print("\nTop-p's nucleus grows quickly with temperature: at high T it admits hundreds of tail tokens.")
print("Min-p's cut is relative to the top token, so it grows more slowly. Top-k ignores the shape entirely.\n")

def repetition_penalty(logits, seen, theta):
    """CTRL / Hugging Face / vLLM style: seen tokens' positive logits are divided by theta, negative ones multiplied."""
    z = np.array(logits, dtype=float)
    for t in set(seen):
        z[t] = z[t] / theta if z[t] > 0 else z[t] * theta
    return z

z = np.array([2.0, 1.5, 0.5, -1.0])
for theta in [1.0, 1.2, 2.0]:
    print(f"theta {theta}: p(token 0 after it was already generated) = {softmax(repetition_penalty(z, [0], theta))[0]:.2f}")

---
## Takeaways
- **Temperature reshapes** the distribution (p^(1/T)); **top-k, top-p and min-p cut** its tail; then one token is sampled. T = 0 is greedy.
- **Order matters:** temperature is applied before truncation, so the same top-p keeps far more tokens at high temperature.
- **Beam search** maximises sequence probability and beats greedy when an early choice is a trap, which suits short closed outputs, not open-ended text.
- Repetition penalties change logits of tokens already seen; strong values also suppress tokens that should repeat.